# 🚀 Entrenamiento de LayoutLMv3 para Extracción de Facturas

Este notebook te guía paso a paso en:
1. ✅ Instalación de dependencias
2. 📁 Configuración de archivos (PDFs + JSONs)
3. 🔧 Generación de dataset con coordenadas normalizadas
4. ✔️ Validación de calidad del dataset
5. 🤖 Entrenamiento de LayoutLMv3
6. 💾 Descarga del modelo entrenado

**Requisitos:**
- Runtime con GPU (recomendado: T4 o superior)
- PDFs de facturas + JSONs con campos anotados

---

## 📋 Paso 1: Instalación de Dependencias

Instalamos todas las librerías necesarias:
- PyMuPDF, pdfplumber: Extracción de PDFs
- OpenCV: Preprocesamiento de imágenes
- PaddleOCR: OCR robusto para PDFs rasterizados
- Transformers, PyTorch: LayoutLMv3

In [ ]:
# Instalar dependencias básicas
!pip install -q PyMuPDF>=1.23.0 pdfplumber>=0.10.0
!pip install -q rapidfuzz>=3.5.0
!pip install -q numpy pandas Pillow
!pip install -q opencv-python>=4.8.0
!pip install -q jsonschema tqdm click colorlog

print("✅ Dependencias básicas instaladas")

In [ ]:
# Instalar PaddleOCR (para PDFs rasterizados)
# NOTA: Esto puede tardar 2-3 minutos
!pip install -q paddlepaddle-gpu>=2.5.0  # GPU version
!pip install -q paddleocr>=2.7.0

print("✅ PaddleOCR instalado (OCR habilitado)")

In [ ]:
# Instalar Transformers y PyTorch (LayoutLMv3)
!pip install -q transformers>=4.35.0
!pip install -q torch torchvision

print("✅ Transformers y PyTorch instalados")

In [ ]:
# Verificar GPU
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU no disponible. El entrenamiento será MUY lento.")
    print("   Ve a Runtime > Change runtime type > GPU")

## 📂 Paso 2: Clonar Repositorio y Subir Archivos

Tienes 2 opciones:
- **Opción A**: Clonar desde GitHub (si tienes el repo público)
- **Opción B**: Subir archivos manualmente a Colab

In [ ]:
# OPCIÓN A: Clonar repositorio desde GitHub
# (Descomentar si tienes el repo en GitHub)

# !git clone https://github.com/tu-usuario/Consultas-Claude.git
# %cd Consultas-Claude

print("Opción A: Comentada. Usa Opción B para subir manualmente.")

In [ ]:
# OPCIÓN B: Crear estructura de directorios y subir archivos
import os
from pathlib import Path

# Crear estructura
!mkdir -p "Datos extraidos de Originales/pdfs"
!mkdir -p "Datos extraidos de Originales/anotaciones"
!mkdir -p src/extractors
!mkdir -p src/matchers
!mkdir -p src/normalizers
!mkdir -p src/validation
!mkdir -p src/exporters
!mkdir -p src/training
!mkdir -p src/preprocessing
!mkdir -p src/cache
!mkdir -p src/utils

print("✅ Estructura de directorios creada")
print("\n📤 Ahora debes:")
print("   1. Subir tus PDFs a: 'Datos extraidos de Originales/pdfs/'")
print("   2. Subir tus JSONs a: 'Datos extraidos de Originales/anotaciones/'")
print("   3. Subir los archivos .py del proyecto a la carpeta src/")
print("\nUsa el explorador de archivos de Colab (icono de carpeta a la izquierda)")

In [ ]:
# Alternativa: Montar Google Drive (si tienes los archivos ahí)
from google.colab import drive

drive.mount('/content/drive')

print("✅ Google Drive montado en /content/drive")
print("\nPuedes copiar archivos desde Drive:")
print("   !cp -r '/content/drive/MyDrive/MisCarpetas/pdfs' 'Datos extraidos de Originales/'")

## 🔧 Paso 3: Generar Dataset con Pipeline Completo

Ejecutamos el script de generación que incluye:
- ✅ Detección automática de PDFs nativos vs rasterizados
- ✅ OCR inteligente cuando es necesario
- ✅ Fuzzy matching avanzado con normalización por tipo de campo
- ✅ Normalización de coordenadas a 0-1000 (LayoutLMv3)
- ✅ Validación automática de calidad

In [ ]:
# Verificar archivos disponibles
import os
from pathlib import Path

pdfs_dir = "Datos extraidos de Originales/pdfs"
json_dir = "Datos extraidos de Originales/anotaciones"

pdfs = list(Path(pdfs_dir).glob("*.pdf"))
jsons = list(Path(json_dir).glob("*.json"))

print(f"📄 PDFs encontrados: {len(pdfs)}")
print(f"📋 JSONs encontrados: {len(jsons)}")

if pdfs:
    print("\nEjemplos de PDFs:")
    for pdf in pdfs[:5]:
        print(f"  - {pdf.name}")

if not pdfs or not jsons:
    print("\n⚠️ ADVERTENCIA: No se encontraron PDFs o JSONs.")
    print("   Asegúrate de subirlos en el paso anterior.")

In [ ]:
# Generar dataset con TODAS las mejoras (Fase 1 + Fase 2)
# NOTA: Esto puede tardar 5-30 minutos dependiendo del número de PDFs

# Si usas el script Python completo:
!python generar_dataset_avanzado.py \
    --pdfs-dir "Datos extraidos de Originales/pdfs" \
    --json-dir "Datos extraidos de Originales/anotaciones" \
    --output-dir dataset_avanzado \
    --usar-ocr \
    --usar-gpu \
    --confianza-minima 0.75

print("\n✅ Dataset generado en: dataset_avanzado/")

## ✔️ Paso 4: Validar Calidad del Dataset

Antes de entrenar, validamos que el dataset no tenga errores:
- Coordenadas fuera de rango
- Bboxes degenerados
- Texto vacío
- Baja confianza
- Aspect ratios extremos

In [ ]:
# Validar dataset generado
!python validar_dataset_layoutlm.py \
    --dataset-dir dataset_avanzado/coordenadas \
    --verbose

print("\n✅ Validación completada")
print("\n⚠️ Si hay errores críticos, revisa el dataset antes de entrenar.")

## 🤖 Paso 5: Entrenar LayoutLMv3

Entrenamos el modelo con los datos preparados.

**Parámetros importantes:**
- `--epochs`: Número de épocas (10-20 para datasets pequeños, 5-10 para grandes)
- `--batch-size`: Tamaño de batch (4 para GPU de 16GB, 2 para 8GB)
- `--lr`: Learning rate (5e-5 es un buen default)

In [ ]:
# Entrenar modelo
# NOTA: Esto puede tardar 30 min - 2 horas dependiendo del tamaño del dataset

!python entrenar_layoutlmv3.py \
    --dataset-dir dataset_avanzado/coordenadas \
    --images-dir dataset_avanzado/imagenes_paginas \
    --output-dir modelo_entrenado \
    --epochs 10 \
    --batch-size 4 \
    --lr 5e-5

print("\n✅ Entrenamiento completado!")

## 📊 Paso 6: Evaluar Resultados del Entrenamiento

Revisamos las métricas de entrenamiento.

In [ ]:
# Ver resumen del modelo entrenado
import json
from pathlib import Path

label_map_path = Path("modelo_entrenado/modelo_final/label_map.json")

if label_map_path.exists():
    with open(label_map_path, 'r') as f:
        label_map = json.load(f)

    print("="*60)
    print("MODELO ENTRENADO - RESUMEN")
    print("="*60)
    print(f"\nNúmero de campos detectados: {len(label_map['campo_to_id'])}")
    print("\nCampos:")
    for campo in sorted(label_map['campo_to_id'].keys()):
        print(f"  - {campo}")
    print()
else:
    print("⚠️ No se encontró el modelo entrenado.")

In [ ]:
# Visualizar logs de entrenamiento (si están disponibles)
import os

logs_dir = "modelo_entrenado/logs"

if os.path.exists(logs_dir):
    print("📊 Logs de entrenamiento disponibles en:", logs_dir)
    print("\nPuedes usar TensorBoard:")
    print("   %load_ext tensorboard")
    print(f"   %tensorboard --logdir {logs_dir}")
else:
    print("ℹ️ No se encontraron logs de TensorBoard.")

## 💾 Paso 7: Descargar Modelo Entrenado

Comprimimos el modelo y lo descargamos para usarlo localmente.

In [ ]:
# Comprimir modelo entrenado
!zip -r modelo_layoutlmv3_facturas.zip modelo_entrenado/modelo_final

print("✅ Modelo comprimido: modelo_layoutlmv3_facturas.zip")
print(f"   Tamaño: {os.path.getsize('modelo_layoutlmv3_facturas.zip') / 1e6:.1f} MB")

In [ ]:
# Descargar modelo
from google.colab import files

files.download('modelo_layoutlmv3_facturas.zip')

print("✅ Descarga iniciada!")
print("\nTambién puedes copiar a Google Drive:")
print("   !cp modelo_layoutlmv3_facturas.zip '/content/drive/MyDrive/'")

## 🎯 Paso 8: Probar el Modelo (Opcional)

Hacemos una prueba rápida del modelo entrenado.

In [ ]:
# Cargar modelo entrenado y hacer predicción de prueba
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
from PIL import Image
import torch
import json

# Cargar modelo
model_path = "modelo_entrenado/modelo_final"
model = LayoutLMv3ForTokenClassification.from_pretrained(model_path)
processor = LayoutLMv3Processor.from_pretrained(model_path)

# Cargar label map
with open(f"{model_path}/label_map.json", 'r') as f:
    label_map = json.load(f)
    id_to_campo = {int(k): v for k, v in label_map['id_to_campo'].items()}

print("✅ Modelo cargado exitosamente!")
print(f"   Campos detectables: {len(id_to_campo)}")

# Función de predicción
def predict_invoice(image_path, words, boxes):
    """
    Predice campos en una factura.

    Args:
        image_path: Ruta a imagen de la factura
        words: Lista de palabras
        boxes: Lista de bboxes (0-1000 scale)
    """
    image = Image.open(image_path).convert("RGB")

    # Procesar
    encoding = processor(
        image,
        words,
        boxes=boxes,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    # Predecir
    with torch.no_grad():
        outputs = model(**encoding)
        predictions = outputs.logits.argmax(-1).squeeze().tolist()

    # Decodificar
    results = []
    for word, box, pred_id in zip(words, boxes, predictions):
        campo = id_to_campo.get(pred_id, 'O')
        if campo != 'O':
            results.append({
                'campo': campo,
                'texto': word,
                'bbox': box
            })

    return results

print("\nFunción de predicción lista. Usa predict_invoice() para predecir.")

## 📚 Recursos Adicionales

### Documentación del Proyecto
- `MEJORAS_LAYOUTLMV3.md`: Fase 1 (mejoras críticas)
- `FASE2_PIPELINE_HIBRIDO.md`: Fase 2 (OCR híbrido)
- `SOPORTE_MULTIPAGINA.md`: Multi-página

### Ajuste de Hiperparámetros

Si los resultados no son buenos, prueba:
- **Aumentar épocas**: 15-20 (si dataset es pequeño)
- **Reducir learning rate**: 3e-5 o 2e-5
- **Aumentar datos**: Agregar más PDFs anotados
- **Ajustar confianza mínima**: 0.8 o 0.9 (más estricto)

### Troubleshooting

**Error: CUDA out of memory**
- Reduce `--batch-size` a 2 o 1
- Usa `--cpu` si es necesario (muy lento)

**Error: PaddleOCR no funciona**
- Verifica instalación: `!pip show paddleocr`
- Usa `--no-ocr` para deshabilitar OCR

**Dataset muy lento**
- Usa cache: El sistema automáticamente cachea OCR
- Regenerar usa cache: ~10x más rápido

---

## 🎉 ¡Listo!

Ahora tienes un modelo LayoutLMv3 entrenado para extraer campos de tus facturas.

Para usar el modelo en producción, descomprímelo y carga con:
```python
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor

model = LayoutLMv3ForTokenClassification.from_pretrained("modelo_final")
processor = LayoutLMv3Processor.from_pretrained("modelo_final")
```